# 02 — Balanced Transformer Preprocessing

This notebook creates the only processed dataset used by the next modelling stages. It removes exact duplicates, samples the same number of reviews from every rating, applies light BERT-family preprocessing, creates a stratified train/validation split, and saves reproducibility metadata. The raw and official test datasets are never modified.

## Experiment configuration

The default now selects `50_000` rows per class. Keep `VALIDATION_PER_CLASS = 2_000` fixed so additional rows increase the training set instead of making validation unnecessarily large. A separate output folder is created automatically.

In [1]:
from pathlib import Path
import json
import sys
from datetime import datetime, timezone

import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.preprocessing.cleaner import preprocess_for_transformer

SAMPLES_PER_CLASS = 50_000  # Total selected rows per rating class.
VALIDATION_PER_CLASS = 2_000  # Keep validation fixed while scaling training data.
RANDOM_STATE = 42
TARGET_COLUMN = 'overall'
TEXT_COLUMN = 'reviewText'
SUMMARY_MAX_WORDS = 40  # Reserve the remaining token budget for reviewText.

INPUT_PATH = PROJECT_ROOT / 'data' / 'raw' / 'train_data.csv'
TEST_INPUT_PATH = PROJECT_ROOT / 'data' / 'raw' / 'test_data.csv'
PRODUCT_METADATA_PATH = PROJECT_ROOT / 'data' / 'raw' / 'title_brand.csv'
OUTPUT_DIR = (
    PROJECT_ROOT
    / 'data'
    / 'processed'
    / f'bert_balanced_{SAMPLES_PER_CLASS}_per_class'
)
OUTPUT_DIR

WindowsPath('D:/Quera/Exercises/amazon-review-sentiment-analysis/data/processed/bert_balanced_50000_per_class')

## Load raw training data

In [2]:
raw_df = pd.read_csv(INPUT_PATH, low_memory=False)
required_columns = {TARGET_COLUMN, TEXT_COLUMN}
missing_columns = required_columns.difference(raw_df.columns)
assert not missing_columns, f'Missing required columns: {sorted(missing_columns)}'

print('Raw shape:', raw_df.shape)
display(raw_df[TARGET_COLUMN].value_counts().sort_index().to_frame('count'))

Raw shape: (838944, 11)


,count
overall,
1,82950
2,56756
3,81239
4,156514
5,461485


## Remove exact duplicates

Exact duplicate rows are removed before sampling so the same observation cannot leak into both train and validation. Duplicate review text with genuinely different records is retained.

In [3]:
raw_rows = len(raw_df)
deduplicated_df = raw_df.drop_duplicates().reset_index(drop=True)
exact_duplicates_removed = raw_rows - len(deduplicated_df)
cleaned_review_candidate = (
    deduplicated_df[TEXT_COLUMN].fillna('').astype(str).map(preprocess_for_transformer)
)
eligible_review_mask = cleaned_review_candidate.str.strip().ne('')
empty_source_reviews_removed = int((~eligible_review_mask).sum())
deduplicated_df = deduplicated_df.loc[eligible_review_mask].reset_index(drop=True)

print('Exact duplicates removed:', exact_duplicates_removed)
print('Missing/blank source reviews removed:', empty_source_reviews_removed)
print('Eligible rows:', len(deduplicated_df))
display(deduplicated_df[TARGET_COLUMN].value_counts().sort_index().to_frame('available'))

Exact duplicates removed: 8327
Missing/blank source reviews removed: 12
Eligible rows: 830605


,available
overall,
1,82247
2,56288
3,80460
4,155141
5,456469


## Create the reproducible balanced subset

In [4]:
if VALIDATION_PER_CLASS >= SAMPLES_PER_CLASS:
    raise ValueError('VALIDATION_PER_CLASS must be smaller than SAMPLES_PER_CLASS')

available_per_class = deduplicated_df[TARGET_COLUMN].value_counts().sort_index()
insufficient = available_per_class[available_per_class < SAMPLES_PER_CLASS]
if not insufficient.empty:
    raise ValueError(
        f'Not enough rows for sampling without replacement: {insufficient.to_dict()}'
    )

# Select validation first so it remains identical when SAMPLES_PER_CLASS changes.
validation_raw = deduplicated_df.groupby(
    TARGET_COLUMN, group_keys=False, sort=True
).sample(n=VALIDATION_PER_CLASS, replace=False, random_state=RANDOM_STATE)
training_pool = deduplicated_df.drop(index=validation_raw.index)
train_raw = training_pool.groupby(
    TARGET_COLUMN, group_keys=False, sort=True
).sample(
    n=SAMPLES_PER_CLASS - VALIDATION_PER_CLASS,
    replace=False,
    random_state=RANDOM_STATE + 1,
)
validation_raw = validation_raw.assign(_split='validation')
train_raw = train_raw.assign(_split='train')
balanced_df = pd.concat([train_raw, validation_raw], ignore_index=True)
balanced_df = balanced_df.sample(frac=1, random_state=RANDOM_STATE).reset_index(drop=True)

display(balanced_df[TARGET_COLUMN].value_counts().sort_index().to_frame('sampled'))
print('Balanced shape:', balanced_df.shape)

,sampled
overall,
1,50000
2,50000
3,50000
4,50000
5,50000


Balanced shape: (250000, 12)


## Apply light preprocessing and build the selected model input

Applied to review and summary: HTML removal, URL removal, whitespace normalization, and safe handling of non-string values.

Intentionally preserved: case, punctuation, stopwords, negation, numbers, emoji, and natural sentence structure.

A controlled ablation on the same train/validation split selected `summary + verified status + helpful-vote bucket + review` as the strongest input. Product title/brand and style/year were tested but reduced validation micro-F1, so they are retained as separate columns rather than injected into the 128-token model input. Tokenization and truncation belong to the future deep-learning notebook.

In [5]:
original_review = balanced_df[TEXT_COLUMN].fillna('').astype(str)
original_summary = balanced_df['summary'].fillna('').astype(str)
balanced_df[TEXT_COLUMN] = original_review.map(preprocess_for_transformer)
balanced_df['summary'] = original_summary.map(preprocess_for_transformer)
changed_review_rows = int(original_review.ne(balanced_df[TEXT_COLUMN]).sum())
changed_summary_rows = int(original_summary.ne(balanced_df['summary']).sum())
empty_text_rows = int(balanced_df[TEXT_COLUMN].eq('').sum())

product_df = pd.read_csv(PRODUCT_METADATA_PATH, low_memory=False)
product_df['_completeness'] = product_df[['title', 'brand']].notna().sum(axis=1)
product_df = (
    product_df.sort_values('_completeness', ascending=False)
    .drop_duplicates('asin', keep='first')
    [['asin', 'title', 'brand']]
)
balanced_df = balanced_df.merge(
    product_df, on='asin', how='left', validate='many_to_one'
)
balanced_df['title'] = balanced_df['title'].fillna('').astype(str).map(preprocess_for_transformer)
balanced_df['brand'] = balanced_df['brand'].fillna('').astype(str).map(preprocess_for_transformer)

balanced_df['verified_str'] = (
    balanced_df['verified'].map({True: 'yes', False: 'no'}).fillna('unknown')
)
numeric_vote = pd.to_numeric(
    balanced_df['vote'].astype('string').str.replace(',', '', regex=False),
    errors='coerce',
)
missing_vote_rows = int(numeric_vote.isna().sum())
balanced_df['vote_bucket'] = pd.cut(
    numeric_vote.fillna(-1),
    bins=[-2, -0.5, 4.5, 9.5, 49.5, np.inf],
    labels=['missing', '0_to_4', '5_to_9', '10_to_49', '50_plus'],
).astype('string')

def limit_words(value, max_words=SUMMARY_MAX_WORDS):
    return ' '.join(str(value).split()[:max_words])

balanced_df['summary_for_model'] = balanced_df['summary'].map(limit_words)
balanced_df['model_input'] = (
    'Verified: ' + balanced_df['verified_str']
    + ' | Helpful votes: ' + balanced_df['vote_bucket'].fillna('missing')
    + ' | Summary: ' + balanced_df['summary_for_model']
    + ' | Review: ' + balanced_df[TEXT_COLUMN]
)

metadata_coverage = balanced_df['title'].ne('').mean()
print('Changed review rows:', changed_review_rows)
print('Changed summary rows:', changed_summary_rows)
print('Empty review rows after preprocessing:', empty_text_rows)
print('Rows retained with missing vote:', missing_vote_rows)
print(f'Product-title coverage: {metadata_coverage:.2%}')
display(balanced_df[[TARGET_COLUMN, 'summary', 'verified_str', 'vote_bucket', 'reviewText', 'model_input']].head(3))

Changed review rows: 157732
Changed summary rows: 5792
Empty review rows after preprocessing: 0
Rows retained with missing vote: 189587
Product-title coverage: 99.99%


,overall,summary,verified_str,vote_bucket,reviewText,model_input
0,1,XPS8900 Boat Anchor,no,missing,Purchased direct from DELL: XPS8900 purchased ...,Verified: no | Helpful votes: missing | Summar...
1,2,no support,yes,missing,Backplate uses 4 holes with a straight edge to...,Verified: yes | Helpful votes: missing | Summa...
2,1,I WOULDNT RECOMMEND THIS UNIT. MAJOR PROBLEMS,yes,0_to_4,Im very disappointed with this product. I thou...,Verified: yes | Helpful votes: 0_to_4 | Summar...


## Validate and create a stratified train/validation split

In [6]:
expected_classes = [1, 2, 3, 4, 5]
class_counts = balanced_df[TARGET_COLUMN].value_counts().sort_index()

assert class_counts.index.tolist() == expected_classes
assert class_counts.eq(SAMPLES_PER_CLASS).all()
assert len(balanced_df) == len(expected_classes) * SAMPLES_PER_CLASS
assert not balanced_df.duplicated().any()
assert empty_text_rows == 0
assert balanced_df['model_input'].notna().all()
assert balanced_df['model_input'].str.strip().ne('').all()
assert metadata_coverage > 0.99

train_df = balanced_df.loc[balanced_df['_split'].eq('train')].drop(columns='_split').reset_index(drop=True)
validation_df = balanced_df.loc[balanced_df['_split'].eq('validation')].drop(columns='_split').reset_index(drop=True)
balanced_df = balanced_df.drop(columns='_split')
assert validation_df[TARGET_COLUMN].value_counts().eq(VALIDATION_PER_CLASS).all()
assert train_df[TARGET_COLUMN].value_counts().eq(SAMPLES_PER_CLASS - VALIDATION_PER_CLASS).all()

split_summary = pd.concat(
    {
        'full': balanced_df[TARGET_COLUMN].value_counts().sort_index(),
        'train': train_df[TARGET_COLUMN].value_counts().sort_index(),
        'validation': validation_df[TARGET_COLUMN].value_counts().sort_index(),
    },
    axis=1,
)
display(split_summary)
print('Validation: PASSED')

,full,train,validation
overall,,,
1,50000,48000,2000
2,50000,48000,2000
3,50000,48000,2000
4,50000,48000,2000
5,50000,48000,2000


Validation: PASSED


## Save isolated processed files and metadata

In [7]:
# Apply the identical feature pipeline to every test row and preserve its order.
test_df = pd.read_csv(TEST_INPUT_PATH, low_memory=False)
test_rows = len(test_df)
test_df['_original_order'] = np.arange(test_rows)
test_df[TEXT_COLUMN] = test_df[TEXT_COLUMN].fillna('').astype(str).map(preprocess_for_transformer)
test_df['summary'] = test_df['summary'].fillna('').astype(str).map(preprocess_for_transformer)
test_df = test_df.merge(product_df, on='asin', how='left', validate='many_to_one', sort=False)
test_df = test_df.sort_values('_original_order').reset_index(drop=True)
test_df['title'] = test_df['title'].fillna('').astype(str).map(preprocess_for_transformer)
test_df['brand'] = test_df['brand'].fillna('').astype(str).map(preprocess_for_transformer)
test_df['verified_str'] = test_df['verified'].map({True: 'yes', False: 'no'}).fillna('unknown')
test_numeric_vote = pd.to_numeric(
    test_df['vote'].astype('string').str.replace(',', '', regex=False), errors='coerce'
)
test_df['vote_bucket'] = pd.cut(
    test_numeric_vote.fillna(-1),
    bins=[-2, -0.5, 4.5, 9.5, 49.5, np.inf],
    labels=['missing', '0_to_4', '5_to_9', '10_to_49', '50_plus'],
).astype('string')
test_df['summary_for_model'] = test_df['summary'].map(limit_words)
test_df['model_input'] = (
    'Verified: ' + test_df['verified_str']
    + ' | Helpful votes: ' + test_df['vote_bucket'].fillna('missing')
    + ' | Summary: ' + test_df['summary_for_model']
    + ' | Review: ' + test_df[TEXT_COLUMN]
)
assert len(test_df) == test_rows
assert test_df['_original_order'].eq(np.arange(test_rows)).all()
assert TARGET_COLUMN not in test_df.columns
assert test_df['model_input'].notna().all()
test_df = test_df.drop(columns='_original_order')

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
balanced_path = OUTPUT_DIR / 'balanced_reviews.csv'
train_path = OUTPUT_DIR / 'train.csv'
validation_path = OUTPUT_DIR / 'validation.csv'
test_path = OUTPUT_DIR / 'test.csv'
metadata_path = OUTPUT_DIR / 'metadata.json'

balanced_df.to_csv(balanced_path, index=False)
train_df.to_csv(train_path, index=False)
validation_df.to_csv(validation_path, index=False)
test_df.to_csv(test_path, index=False)

metadata = {
    'created_at_utc': datetime.now(timezone.utc).isoformat(),
    'source_file': str(INPUT_PATH.relative_to(PROJECT_ROOT)),
    'source_test_file': str(TEST_INPUT_PATH.relative_to(PROJECT_ROOT)),
    'samples_per_class': SAMPLES_PER_CLASS,
    'random_state': RANDOM_STATE,
    'validation_per_class': VALIDATION_PER_CLASS,
    'raw_rows': raw_rows,
    'exact_duplicates_removed': exact_duplicates_removed,
    'empty_source_reviews_removed': empty_source_reviews_removed,
    'balanced_rows': len(balanced_df),
    'train_rows': len(train_df),
    'validation_rows': len(validation_df),
    'test_rows': len(test_df),
    'class_distribution': {str(k): int(v) for k, v in class_counts.items()},
    'changed_review_rows': changed_review_rows,
    'changed_summary_rows': changed_summary_rows,
    'product_title_coverage': metadata_coverage,
    'preprocessing': ['remove_html', 'remove_urls', 'normalize_whitespace'],
    'model_input_column': 'model_input',
    'summary_max_words': SUMMARY_MAX_WORDS,
    'missing_vote_policy': 'retain_row_and_encode_as_missing',
    'missing_vote_rows': missing_vote_rows,
    'model_input_fields': ['verified_str', 'vote_bucket', 'summary_for_model', 'reviewText'],
    'metadata_considered_but_excluded_from_model_input': [
        'title', 'brand', 'style', 'reviewTime', 'reviewerID',
        'reviewerName', 'asin', 'unixReviewTime'
    ],
    'preserved_for_transformers': [
        'case', 'punctuation', 'stopwords', 'negation', 'numbers', 'emoji'
    ],
    'columns': balanced_df.columns.tolist(),
}
metadata_path.write_text(
    json.dumps(metadata, indent=2, ensure_ascii=False) + '\n',
    encoding='utf-8',
)

print('Saved:')
for path in [balanced_path, train_path, validation_path, test_path, metadata_path]:
    print('-', path)

Saved:
- D:\Quera\Exercises\amazon-review-sentiment-analysis\data\processed\bert_balanced_50000_per_class\balanced_reviews.csv
- D:\Quera\Exercises\amazon-review-sentiment-analysis\data\processed\bert_balanced_50000_per_class\train.csv
- D:\Quera\Exercises\amazon-review-sentiment-analysis\data\processed\bert_balanced_50000_per_class\validation.csv
- D:\Quera\Exercises\amazon-review-sentiment-analysis\data\processed\bert_balanced_50000_per_class\test.csv
- D:\Quera\Exercises\amazon-review-sentiment-analysis\data\processed\bert_balanced_50000_per_class\metadata.json


## Output contract

The generated folder is isolated by experiment size. With the current setting it is `data/processed/bert_balanced_50000_per_class/`. Training notebooks use `train.csv` and `validation.csv`; `test.csv` contains every official test row in its original order with the identical feature pipeline and remains label-free until final inference.